# 🎬 Benchmark: uniVIP — Unified Arbitrary-Time Video Frame Interpolation

> **Paper:** [Unified Arbitrary-Time Video Frame Interpolation and Prediction](https://arxiv.org/pdf/2503.02316) (ICASSP 2025)  
> **Dataset:** ATD-12K (Animation Triplet Dataset) — 540p test split  
> **Metrics:** 17 standardized metrics (PSNR, SSIM, LPIPS, IE, Arbitrary-t, Speed, VRAM, ...)  
> **Environment:** Kaggle Notebook · GPU T4 x2 / P100

---

## 📋 Danh Sách 17 Chỉ Số Benchmark

| # | Chỉ Số | Trạng Thái |
|---|--------|:-----------:|
| 1 | Chamfer Distance (CD) | ❌ N/A |
| 2 | Weighted CD (WCD) | ❌ N/A |
| 3 | **LPIPS** | ✅ |
| 4 | EMD | ❌ N/A |
| 5 | RRLD | ❌ N/A |
| 6 | **PSNR** | ✅ |
| 7 | **SSIM** | ✅ |
| 8 | IE (MAE/RMSE) | ⚠️ |
| 9 | FVD | ⚠️ N/A (needs video seq) |
| 10 | Warping Error | ⚠️ N/A (needs video seq) |
| 11 | Hausdorff Distance | ❌ N/A |
| 12 | Line IoU | ❌ N/A |
| 13 | Disconnection Rate | ❌ N/A |
| 14 | **Arbitrary t Support** | ✅ |
| 15 | **Inference Speed** | ✅ |
| 16 | **VRAM Peak** | ✅ |
| 17 | Fine-tuning Speed | ⚠️ |

---
## CELL 1 — Thiết Lập Môi Trường & Cài Đặt Thư Viện

Kiểm tra GPU, cài CuPy (bắt buộc cho softmax-splatting) và các thư viện phụ.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 1: Thiết Lập Môi Trường
# ──────────────────────────────────────────────────────────────────────────────

# 1.1 Kiểm tra GPU
print('=' * 60)
print('  GPU INFO')
print('=' * 60)
!nvidia-smi
!nvcc --version | grep 'release'

In [ ]:
# 1.2 Phát hiện CUDA version và cài CuPy tương ứng
import subprocess
import sys

def get_cuda_major_version():
    try:
        out = subprocess.check_output(['nvcc', '--version']).decode()
        for line in out.split('\n'):
            if 'release' in line:
                # Ví dụ: 'Cuda compilation tools, release 11.8, V11.8.89'
                parts = line.split('release')[1].strip().split(',')
                version_str = parts[0].strip()  # '11.8'
                major = int(version_str.split('.')[0])
                return major
    except Exception:
        return None

cuda_major = get_cuda_major_version()
print(f'Detected CUDA major version: {cuda_major}')

if cuda_major == 11:
    cupy_pkg = 'cupy-cuda11x'
elif cuda_major == 12:
    cupy_pkg = 'cupy-cuda12x'
elif cuda_major == 10:
    cupy_pkg = 'cupy-cuda102==9.4.0'
else:
    cupy_pkg = 'cupy-cuda11x'  # fallback
    print(f'[WARNING] Unknown CUDA version, defaulting to cupy-cuda11x')

print(f'Installing: {cupy_pkg}')

In [ ]:
# 1.3 Cài đặt tất cả thư viện cần thiết
!pip install -q lpips scikit-image tqdm pandas matplotlib tabulate
!pip install -q {cupy_pkg}

In [ ]:
# 1.4 Verify imports
import torch
import numpy as np
import cv2
import lpips
import pandas as pd
import matplotlib

print('Package versions:')
print(f'  PyTorch:     {torch.__version__}')
print(f'  NumPy:       {np.__version__}')
print(f'  OpenCV:      {cv2.__version__}')
print(f'  Pandas:      {pd.__version__}')
print(f'  Matplotlib:  {matplotlib.__version__}')
print()

# Verify CuPy
try:
    import cupy
    print(f'  CuPy:        {cupy.__version__} ✅')
except ImportError:
    print('  CuPy:        NOT FOUND ❌ — softmax-splatting sẽ không hoạt động!')
print()

# GPU check
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} | {props.total_memory / 1e9:.1f} GB VRAM')
else:
    print('  ❌ No GPU detected! Please enable GPU in Notebook settings.')

---
## CELL 2 — Chuẩn Bị Source Code & Nạp Model

Thiết lập đường dẫn Kaggle Dataset, sys.path, và load Pipeline uniVIP.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 2: Chuẩn Bị Source Code & Pretrained Checkpoints
# ──────────────────────────────────────────────────────────────────────────────
import sys
import os

# ─── CẤU HÌNH ĐƯỜNG DẪN ─────────────────────────────────────────────────────
# Thay đổi các giá trị dưới đây nếu cần

MODEL_ROOT = "/kaggle/input/univip-model/uniVIP"
# ↑ Đường dẫn tới thư mục uniVIP trong Kaggle Dataset 'univip-model'
# Nếu dùng Git clone: MODEL_ROOT = "/kaggle/working/uniVIP"

DATASET_ROOT = "/kaggle/input/atd12k-benchmark/test_2k_540p"
# ↑ Đường dẫn tới thư mục test_2k_540p trong Kaggle Dataset 'atd12k-benchmark'

OUTPUT_DIR = "/kaggle/working/benchmark_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Kiểm tra đường dẫn tồn tại
assert os.path.isdir(MODEL_ROOT),   f'❌ MODEL_ROOT không tồn tại: {MODEL_ROOT}'
assert os.path.isdir(DATASET_ROOT), f'❌ DATASET_ROOT không tồn tại: {DATASET_ROOT}'
print(f'✅ MODEL_ROOT:   {MODEL_ROOT}')
print(f'✅ DATASET_ROOT: {DATASET_ROOT}')
print(f'✅ OUTPUT_DIR:   {OUTPUT_DIR}')

# ─── THÊM SOURCE CODE VÀO PYTHON PATH ──────────────────────────────────────
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)
print(f'\n✅ sys.path updated: {MODEL_ROOT}')

# Test import
try:
    import core
    print('✅ core module import OK')
except ImportError as e:
    print(f'❌ core import FAILED: {e}')
    print('  Kiểm tra lại MODEL_ROOT và cấu trúc thư mục.')

In [ ]:
# ─── LOAD MODEL ──────────────────────────────────────────────────────────────
import torch
from core.pipeline import Pipeline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ─── CHỌN VARIANT ────────────────────────────────────────────────────────────
# False = uniVIP-B (Base, ~6.3 MB checkpoint, ít VRAM hơn)
# True  = uniVIP-L (Large, ~25 MB checkpoint, chính xác hơn)
USE_LARGE_MODEL = False

MODEL_VARIANT = 'uniVIP-L' if USE_LARGE_MODEL else 'uniVIP-B'
MODEL_FILE    = os.path.join(MODEL_ROOT, 'checkpoints', f'{MODEL_VARIANT}.pkl')

assert os.path.isfile(MODEL_FILE), f'❌ Checkpoint không tồn tại: {MODEL_FILE}'
print(f'✅ Checkpoint tìm thấy: {MODEL_FILE} ({os.path.getsize(MODEL_FILE) / 1e6:.1f} MB)')

# ─── THIẾT LẬP CUDNN ─────────────────────────────────────────────────────────
torch.set_grad_enabled(False)
if torch.cuda.is_available():
    torch.backends.cudnn.enabled   = True
    torch.backends.cudnn.benchmark = True

# ─── KHỞI TẠO PIPELINE ───────────────────────────────────────────────────────
model_cfg = dict(
    load_pretrain = True,
    large_model   = USE_LARGE_MODEL,
    model_file    = MODEL_FILE,
)

ppl = Pipeline(model_cfg)
ppl.eval()

# Đếm số tham số
n_params = sum(p.numel() for p in ppl.model.parameters())
print(f'\n✅ Model {MODEL_VARIANT} đã được nạp thành công!')
print(f'   Tham số: {n_params / 1e6:.2f}M')
print(f'   Device:  {DEVICE}')

---
## CELL 3 — Dataloader / Hàm Đọc Dữ Liệu Triplet ATD-12K

Quét thư mục triplet, tiền xử lý ảnh theo đúng format uniVIP:
- **frame1.png** → `img0` (t=0, input)
- **frame3.png** → `img1` (t=1, input)
- **frame2.png** → `gt` (t=0.5, Ground Truth)
- Padding kích thước chia hết cho **64** trước inference

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 3: Dataloader ATD-12K Triplet
# ──────────────────────────────────────────────────────────────────────────────
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# uniVIP yêu cầu kích thước H và W chia hết cho 64
DIVISOR = 64


class ATD12KDataset(Dataset):
    """PyTorch Dataset cho dữ liệu ATD-12K triplet.
    
    Quy ước:
        frame1.png → img0 (t=0, input đầu vào)
        frame3.png → img1 (t=1, input đầu vào)
        frame2.png → gt   (t=0.5, Ground Truth cần dự đoán)
    
    Tiền xử lý:
        - Đọc ảnh bằng OpenCV → BGR format
        - Chuyển HWC → CHW
        - Normalize [0, 255] → [0.0, 1.0] float32
        - KHÔNG pad trong Dataset (pad thực hiện trong pipeline chính)
    """
    
    def __init__(self, root: str, max_samples: int = None):
        """
        Args:
            root:        Đường dẫn tới thư mục test_2k_540p/
            max_samples: Giới hạn số clip (None = dùng tất cả)
        """
        self.root = Path(root)
        if not self.root.is_dir():
            raise ValueError(f'Dataset root không tồn tại: {root}')
        
        # Quét tất cả thư mục con có đủ 3 file frame
        self.clips = sorted([
            d for d in self.root.iterdir()
            if d.is_dir()
            and (d / 'frame1.png').exists()
            and (d / 'frame2.png').exists()
            and (d / 'frame3.png').exists()
        ])
        
        # Báo lỗi nếu không tìm thấy clip nào
        if len(self.clips) == 0:
            raise RuntimeError(
                f'Không tìm thấy clip nào trong {root}!\n'
                f'Mỗi thư mục con phải có frame1.png, frame2.png, frame3.png'
            )
        
        # Giới hạn số mẫu nếu cần
        if max_samples is not None:
            self.clips = self.clips[:max_samples]
        
        print(f'📦 ATD-12K Dataset: {len(self.clips)} clips tìm thấy trong "{root}"')
    
    def __len__(self):
        return len(self.clips)
    
    def __getitem__(self, idx):
        clip_dir = self.clips[idx]
        
        # Đọc 3 ảnh (OpenCV → BGR format)
        img0_bgr = cv2.imread(str(clip_dir / 'frame1.png'))
        img1_bgr = cv2.imread(str(clip_dir / 'frame3.png'))
        gt_bgr   = cv2.imread(str(clip_dir / 'frame2.png'))
        
        # Kiểm tra đọc thành công
        for img, name in [(img0_bgr, 'frame1.png'), (img1_bgr, 'frame3.png'), (gt_bgr, 'frame2.png')]:
            if img is None:
                raise IOError(f'Không đọc được ảnh: {clip_dir / name}')
        
        # Kiểm tra kích thước đồng nhất
        assert img0_bgr.shape == img1_bgr.shape == gt_bgr.shape, \
            f'Kích thước không khớp trong clip {clip_dir.name}'
        
        h, w = img0_bgr.shape[:2]
        
        # Hàm chuyển numpy HWC → tensor CHW [0,1]
        def to_tensor(img_bgr):
            return torch.from_numpy(
                img_bgr.transpose(2, 0, 1).astype(np.float32) / 255.0
            )  # shape: (C, H, W), dtype: float32
        
        return {
            'img0':  to_tensor(img0_bgr),   # (3, H, W) BGR float [0,1]
            'img1':  to_tensor(img1_bgr),   # (3, H, W) BGR float [0,1]
            'gt':    to_tensor(gt_bgr),     # (3, H, W) BGR float [0,1]
            'h':     h,
            'w':     w,
            'clip':  str(clip_dir.name),
        }


def pad_to_divisor(tensor: torch.Tensor, divisor: int = 64):
    """Pad tensor (B, C, H, W) sao cho H và W chia hết cho divisor.
    
    Padding bằng giá trị 0.5 (trung gian), giúp tránh artifacts ở biên.
    Trả về tensor đã pad và kích thước gốc (h, w).
    """
    _, _, h, w = tensor.shape
    ph = ((h - 1) // divisor + 1) * divisor
    pw = ((w - 1) // divisor + 1) * divisor
    if ph == h and pw == w:
        return tensor, h, w  # Không cần pad
    padding = (0, pw - w, 0, ph - h)  # (left, right, top, bottom)
    padded = F.pad(tensor, padding, mode='constant', value=0.5)
    return padded, h, w


# ─── TEST DATALOADER ─────────────────────────────────────────────────────────
# MAX_SAMPLES = 10  # Uncomment để test nhanh với 10 clip đầu
MAX_SAMPLES = None  # None = dùng toàn bộ dataset

dataset = ATD12KDataset(DATASET_ROOT, max_samples=MAX_SAMPLES)
loader  = DataLoader(dataset, batch_size=1, shuffle=False,
                     num_workers=2, pin_memory=True)

# Preview một sample
sample = dataset[0]
print(f'\n📊 Sample preview (clip: {sample["clip"]})')
print(f'   img0 shape:  {sample["img0"].shape}  (C, H, W)')
print(f'   img1 shape:  {sample["img1"].shape}')
print(f'   gt shape:    {sample["gt"].shape}')
print(f'   Resolution:  {sample["w"]} x {sample["h"]}')
print(f'   Value range: [{sample["img0"].min():.3f}, {sample["img0"].max():.3f}]')
print(f'\n✅ Dataloader sẵn sàng: {len(dataset)} samples')

---
## CELL 4 — Module Tính Toán Metrics Chuẩn

Định nghĩa các hàm tính toán đầy đủ cho metrics ✅ và ⚠️:
- **PSNR** (✅): Peak Signal-to-Noise Ratio
- **SSIM** (✅): Structural Similarity Index
- **LPIPS** (✅): Learned Perceptual Image Patch Similarity
- **IE MAE/RMSE** (⚠️): Interpolation Error
- **Arbitrary t** (✅): PSNR tại t=0.25, 0.5, 0.75
- **Inference Speed** (✅): ms/frame, FPS (warmup-free)
- **VRAM Peak** (✅): MB

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 4: Module Tính Toán Metrics Chuẩn
# ──────────────────────────────────────────────────────────────────────────────
import math
import time
import lpips as lpips_lib
import torch
import numpy as np
from skimage.metrics import structural_similarity as compare_ssim

# ─── KHỞI TẠO LPIPS ──────────────────────────────────────────────────────────
print('Đang tải LPIPS model (AlexNet)...')
lpips_fn = lpips_lib.LPIPS(net='alex', verbose=False).to(DEVICE)
lpips_fn.eval()
print('✅ LPIPS (AlexNet) sẵn sàng')


# ──────────────────────────────────────────────────────────────────────────────
# METRIC 1: PSNR (✅)
# ──────────────────────────────────────────────────────────────────────────────
def compute_psnr(pred: torch.Tensor, gt: torch.Tensor) -> float:
    """Tính PSNR (Peak Signal-to-Noise Ratio) theo dB.
    
    Args:
        pred: Tensor (C, H, W) float [0, 1], bất kỳ channel order nào
        gt:   Tensor (C, H, W) float [0, 1]
    
    Returns:
        float: PSNR (dB), càng cao càng tốt (thường 30–45 dB cho ảnh tốt)
    """
    mse = torch.mean((pred.float() - gt.float()) ** 2).item()
    if mse < 1e-10:
        return float('inf')  # Ảnh giống hệt
    return -10.0 * math.log10(mse)


# ──────────────────────────────────────────────────────────────────────────────
# METRIC 2: SSIM (✅)
# ──────────────────────────────────────────────────────────────────────────────
def compute_ssim(pred: torch.Tensor, gt: torch.Tensor) -> float:
    """Tính SSIM (Structural Similarity Index).
    
    QUAN TRỌNG: uniVIP output BGR. Phải chuyển sang RGB trước khi tính SSIM.
    
    Args:
        pred: Tensor (C, H, W) BGR float [0, 1]
        gt:   Tensor (C, H, W) BGR float [0, 1]
    
    Returns:
        float: SSIM ∈ [0, 1], càng cao càng tốt
    """
    # Chuyển BGR tensor → RGB numpy HWC
    pred_np = pred.cpu().numpy().transpose(1, 2, 0)[:, :, ::-1]  # BGR→RGB, HWC
    gt_np   = gt.cpu().numpy().transpose(1, 2, 0)[:, :, ::-1]
    pred_np = np.clip(pred_np, 0.0, 1.0)
    gt_np   = np.clip(gt_np,   0.0, 1.0)
    return float(compare_ssim(pred_np, gt_np, data_range=1.0, channel_axis=2))


# ──────────────────────────────────────────────────────────────────────────────
# METRIC 3: LPIPS (✅)
# ──────────────────────────────────────────────────────────────────────────────
def compute_lpips(pred: torch.Tensor, gt: torch.Tensor, fn) -> float:
    """Tính LPIPS (Learned Perceptual Image Patch Similarity).
    
    LPIPS mong đợi tensor RGB chuẩn hóa trong [-1, 1].
    uniVIP output là BGR [0, 1] → cần đảo channel và scale.
    
    Args:
        pred: Tensor (C, H, W) BGR float [0, 1]
        gt:   Tensor (C, H, W) BGR float [0, 1]
        fn:   LPIPS model instance
    
    Returns:
        float: LPIPS ∈ [0, 1], càng thấp càng tốt
    """
    # BGR → RGB: đảo chiều channel [B,G,R] → [R,G,B]
    pred_rgb = pred[[2, 1, 0], :, :].unsqueeze(0).to(DEVICE)  # (1, 3, H, W) RGB
    gt_rgb   = gt[[2, 1, 0], :, :].unsqueeze(0).to(DEVICE)
    # Normalize [0,1] → [-1,1]
    pred_rgb = pred_rgb * 2.0 - 1.0
    gt_rgb   = gt_rgb   * 2.0 - 1.0
    with torch.no_grad():
        dist = fn(pred_rgb, gt_rgb)
    return float(dist.item())


# ──────────────────────────────────────────────────────────────────────────────
# METRIC 4: IE — Interpolation Error (⚠️)
# ──────────────────────────────────────────────────────────────────────────────
def compute_ie(pred: torch.Tensor, gt: torch.Tensor) -> dict:
    """Tính Interpolation Error (MAE và RMSE).
    
    Args:
        pred: Tensor (C, H, W) float [0, 1]
        gt:   Tensor (C, H, W) float [0, 1]
    
    Returns:
        dict: {'IE_MAE': float, 'IE_RMSE': float}
              MAE  = Mean Absolute Error, càng thấp càng tốt
              RMSE = Root Mean Squared Error, càng thấp càng tốt
    """
    diff = pred.float() - gt.float()
    mae  = torch.mean(torch.abs(diff)).item()
    rmse = torch.sqrt(torch.mean(diff ** 2)).item()
    return {'IE_MAE': mae, 'IE_RMSE': rmse}


# ──────────────────────────────────────────────────────────────────────────────
# METRIC 5: Arbitrary t Support (✅)
# ──────────────────────────────────────────────────────────────────────────────
def test_arbitrary_t(ppl, img0: torch.Tensor, img1: torch.Tensor,
                     h: int, w: int,
                     t_values: list = [0.25, 0.5, 0.75]) -> dict:
    """Kiểm tra khả năng sinh frame tại nhiều mốc thời gian t.
    
    Đây là tính năng độc đáo của uniVIP: t ∈ (0,1) là interpolation,
    t > 1 là prediction (future frame).
    
    Args:
        ppl:      Pipeline object
        img0:     Tensor (1, C, H', W') đã pad, trên DEVICE
        img1:     Tensor (1, C, H', W') đã pad, trên DEVICE
        h, w:     Kích thước gốc trước pad
        t_values: Danh sách các giá trị t cần kiểm tra
    
    Returns:
        dict: Mapping t → (pred_tensor, success)
    """
    results = {}
    for t in t_values:
        try:
            with torch.no_grad():
                pred, _ = ppl.model(img0, img1, time_period=float(t))
                pred = pred[:, :, :h, :w]  # Crop về size gốc
            results[f't={t:.2f}'] = {'pred': pred[0], 'success': True, 'error': None}
        except Exception as e:
            results[f't={t:.2f}'] = {'pred': None, 'success': False, 'error': str(e)}
    return results


# ──────────────────────────────────────────────────────────────────────────────
# METRIC 6: Inference Speed (✅)
# ──────────────────────────────────────────────────────────────────────────────
def measure_inference_speed(ppl, img0: torch.Tensor, img1: torch.Tensor,
                             n_warmup: int = 5, n_repeat: int = 30) -> dict:
    """Đo inference latency chính xác, loại bỏ GPU warmup.
    
    Quy trình:
        1. Warmup n_warmup lần (không tính thời gian)
        2. Synchronize GPU
        3. Đo n_repeat lần → tính trung bình
    
    Args:
        ppl:      Pipeline object
        img0:     Tensor đã pad trên DEVICE
        img1:     Tensor đã pad trên DEVICE
        n_warmup: Số lần warmup
        n_repeat: Số lần đo
    
    Returns:
        dict: {'Speed_ms': ms/frame, 'Speed_FPS': FPS}
    """
    # Giai đoạn warmup — bỏ qua thời gian
    for _ in range(n_warmup):
        with torch.no_grad():
            ppl.model(img0, img1, time_period=0.5)
    
    # Giai đoạn đo chính xác
    torch.cuda.synchronize()
    t_start = time.perf_counter()
    for _ in range(n_repeat):
        with torch.no_grad():
            ppl.model(img0, img1, time_period=0.5)
    torch.cuda.synchronize()
    t_end = time.perf_counter()
    
    avg_ms  = (t_end - t_start) / n_repeat * 1000.0
    avg_fps = 1000.0 / avg_ms
    return {'Speed_ms': round(avg_ms, 3), 'Speed_FPS': round(avg_fps, 2)}


# ──────────────────────────────────────────────────────────────────────────────
# METRIC 7: VRAM Peak (✅)
# ──────────────────────────────────────────────────────────────────────────────
def measure_vram_peak(ppl, img0: torch.Tensor, img1: torch.Tensor) -> float:
    """Đo VRAM peak (MB) trong quá trình một lần inference.
    
    Returns:
        float: VRAM peak (MB)
    """
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    with torch.no_grad():
        ppl.model(img0, img1, time_period=0.5)
    torch.cuda.synchronize()
    peak_bytes = torch.cuda.max_memory_allocated()
    return round(peak_bytes / 1e6, 2)  # Chuyển sang MB


# ─── TEST NHANH CÁC HÀM METRICS ──────────────────────────────────────────────
print('\n📐 Test nhanh các hàm metric...')
dummy_pred = torch.rand(3, 256, 256)
dummy_gt   = torch.rand(3, 256, 256)

psnr_val = compute_psnr(dummy_pred, dummy_gt)
ssim_val = compute_ssim(dummy_pred, dummy_gt)
lpips_val = compute_lpips(dummy_pred, dummy_gt, lpips_fn)
ie_val   = compute_ie(dummy_pred, dummy_gt)

print(f'   PSNR:      {psnr_val:.4f} dB')
print(f'   SSIM:      {ssim_val:.6f}')
print(f'   LPIPS:     {lpips_val:.6f}')
print(f'   IE (MAE):  {ie_val["IE_MAE"]:.6f}')
print(f'   IE (RMSE): {ie_val["IE_RMSE"]:.6f}')
print('\n✅ Tất cả hàm metric hoạt động bình thường!')

---
## CELL 5 — Pipeline Benchmark Tự Động & Xuất Báo Cáo 17 Chỉ Số

Chạy vòng lặp benchmark trên toàn bộ dataset, tổng hợp và xuất kết quả.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5A: CẤU HÌNH BENCHMARK
# ──────────────────────────────────────────────────────────────────────────────

# ─── Tùy chỉnh các tham số dưới đây ─────────────────────────────────────────
BENCH_MAX_SAMPLES   = None    # None = toàn bộ dataset | số nguyên = test nhanh
BENCH_TIME_PERIOD   = 0.5     # t=0.5 là interpolation (GT có sẵn trong ATD-12K)
BENCH_FIXED_PYR_LVL = 5       # Fixed pyramid level (recommend từ paper uniVIP)
                               # Thử 3 nếu bị OOM
BENCH_SPEED_WARMUP  = 5       # Số lần warmup GPU trước khi đo speed
BENCH_SPEED_REPEAT  = 20      # Số lần lặp để đo speed chính xác
BENCH_VIZ_SAMPLES   = 4       # Số mẫu visualization (0 = tắt visualization)
BENCH_USE_TRY_EXCEPT = True   # Bọc inference trong try-except để không bị dừng

# Arbitrary t values để kiểm tra (chỉ đo PSNR tại t=0.5 vì chỉ GT ở t=0.5)
ARBITRARY_T_VALUES = [0.25, 0.5, 0.75, 1.5]  # 1.5 = prediction

print('⚙️  Cấu hình Benchmark:')
print(f'   Model:           {MODEL_VARIANT}')
print(f'   Max samples:     {BENCH_MAX_SAMPLES if BENCH_MAX_SAMPLES else "All"}')
print(f'   Time period:     t={BENCH_TIME_PERIOD}')
print(f'   Pyramid level:   {BENCH_FIXED_PYR_LVL}')
print(f'   Speed warmup:    {BENCH_SPEED_WARMUP}x')
print(f'   Speed repeats:   {BENCH_SPEED_REPEAT}x')
print(f'   Viz samples:     {BENCH_VIZ_SAMPLES}')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5B: ĐO INFERENCE SPEED CHÍNH XÁC (WARMUP-FREE)
# Chạy trước trên 1 sample để có baseline speed
# ──────────────────────────────────────────────────────────────────────────────
print('🔥 Đo Inference Speed (warmup-free)...')
print(f'   Warmup: {BENCH_SPEED_WARMUP}x | Measure: {BENCH_SPEED_REPEAT}x')

sample_0       = dataset[0]
img0_speed     = sample_0['img0'].unsqueeze(0).to(DEVICE)  # (1, C, H, W)
img1_speed     = sample_0['img1'].unsqueeze(0).to(DEVICE)
img0_speed_pad, sh, sw = pad_to_divisor(img0_speed, DIVISOR)
img1_speed_pad, _,  _  = pad_to_divisor(img1_speed, DIVISOR)

speed_result = measure_inference_speed(
    ppl, img0_speed_pad, img1_speed_pad,
    n_warmup=BENCH_SPEED_WARMUP,
    n_repeat=BENCH_SPEED_REPEAT
)

GLOBAL_SPEED_MS  = speed_result['Speed_ms']
GLOBAL_SPEED_FPS = speed_result['Speed_FPS']
print(f'\n✅ Inference Speed:')
print(f'   {GLOBAL_SPEED_MS:.2f} ms/frame')
print(f'   {GLOBAL_SPEED_FPS:.1f} FPS')
print(f'   (Ảnh đầu vào: {sw}x{sh} → padded: {img0_speed_pad.shape[-1]}x{img0_speed_pad.shape[-2]})')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5C: VÒNG LẶP BENCHMARK CHÍNH
# ──────────────────────────────────────────────────────────────────────────────
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Tạo dataset & loader cho benchmark
bench_dataset = ATD12KDataset(DATASET_ROOT, max_samples=BENCH_MAX_SAMPLES)
bench_loader  = DataLoader(bench_dataset, batch_size=1, shuffle=False,
                           num_workers=2, pin_memory=True)

results_list = []  # Danh sách kết quả mỗi clip
viz_frames   = []  # Frames để visualization
error_clips  = []  # Clips bị lỗi

print(f'\n🚀 Bắt đầu Benchmark uniVIP ({MODEL_VARIANT})')
print(f'   Dataset: {len(bench_dataset)} clips')
print(f'   Output:  {OUTPUT_DIR}')
print()

for batch_idx, batch in enumerate(tqdm(bench_loader, desc=f'Benchmark {MODEL_VARIANT}', unit='clip')):
    
    clip_name = batch['clip'][0]
    orig_h    = int(batch['h'][0].item())
    orig_w    = int(batch['w'][0].item())
    
    # Đưa tensors lên GPU
    img0_raw = batch['img0'].to(DEVICE, non_blocking=True)  # (1, C, H, W) BGR [0,1]
    img1_raw = batch['img1'].to(DEVICE, non_blocking=True)
    gt_raw   = batch['gt'].to(DEVICE,   non_blocking=True)
    
    # Padding chia hết cho 64
    img0_pad, ph, pw = pad_to_divisor(img0_raw, DIVISOR)
    img1_pad, _,  _  = pad_to_divisor(img1_raw, DIVISOR)
    
    # ── Inference với đo thời gian ──────────────────────────────────────────
    pred = None
    latency_ms  = None
    vram_mb     = None
    infer_error = None
    
    try:
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        
        with torch.no_grad():
            # Thử với fixed_pyr_level=5 (khuyến nghị từ paper)
            try:
                pred, info_dict = ppl.model(
                    img0_pad, img1_pad,
                    time_period=float(BENCH_TIME_PERIOD),
                    skip_pad=True,
                    fixed_pyr_level=BENCH_FIXED_PYR_LVL
                )
            except TypeError:
                # Fallback: bỏ tham số skip_pad và fixed_pyr_level nếu version cũ
                pred, info_dict = ppl.model(
                    img0_pad, img1_pad,
                    time_period=float(BENCH_TIME_PERIOD)
                )
        
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        
        latency_ms = (t1 - t0) * 1000.0
        vram_mb    = torch.cuda.max_memory_allocated() / 1e6
        
        # Crop về kích thước gốc
        pred = pred[:, :, :orig_h, :orig_w]
    
    except Exception as e:
        infer_error = str(e)
        error_clips.append({'clip': clip_name, 'error': infer_error})
        if not BENCH_USE_TRY_EXCEPT:
            raise
    
    # ── Tính Metrics ────────────────────────────────────────────────────────
    if pred is not None:
        pred_s = pred[0]    # (C, H, W)
        gt_s   = gt_raw[0]  # (C, H, W)
        
        psnr_val = compute_psnr(pred_s, gt_s)
        ssim_val = compute_ssim(pred_s, gt_s)
        lpips_val = compute_lpips(pred_s, gt_s, lpips_fn)
        ie_val    = compute_ie(pred_s, gt_s)
        
        # Lưu frame để visualize (chỉ lưu một số đầu tiên)
        if len(viz_frames) < BENCH_VIZ_SAMPLES:
            viz_frames.append({
                'clip':  clip_name,
                'img0':  img0_raw[0].cpu().numpy().transpose(1, 2, 0)[:, :, ::-1],  # RGB
                'img1':  img1_raw[0].cpu().numpy().transpose(1, 2, 0)[:, :, ::-1],
                'pred':  pred_s.cpu().numpy().transpose(1, 2, 0)[:, :, ::-1],
                'gt':    gt_s.cpu().numpy().transpose(1, 2, 0)[:, :, ::-1],
                'psnr':  psnr_val,
                'ssim':  ssim_val,
                'lpips': lpips_val,
            })
    else:
        # Lỗi inference: điền None
        psnr_val  = None
        ssim_val  = None
        lpips_val = None
        ie_val    = {'IE_MAE': None, 'IE_RMSE': None}
    
    # ── Tổng Hợp Row Kết Quả (17 chỉ số + metadata) ──────────────────────────
    row = {
        # ── Metadata ──
        'clip_name':   clip_name,
        'model':       MODEL_VARIANT,
        'resolution':  f'{orig_w}x{orig_h}',
        'status':      'ERROR' if infer_error else 'OK',
        'error_msg':   infer_error if infer_error else '',
        
        # ── 17 Metrics Chuẩn Hóa ──
        # 1. Chamfer Distance (CD) ❌
        'CD':                  'N/A',
        # 2. Weighted CD (WCD) ❌
        'WCD':                 'N/A',
        # 3. LPIPS ✅
        'LPIPS':               round(lpips_val, 6) if lpips_val is not None else None,
        # 4. EMD ❌
        'EMD':                 'N/A',
        # 5. RRLD ❌
        'RRLD':                'N/A',
        # 6. PSNR ✅
        'PSNR':                round(psnr_val, 4) if psnr_val is not None else None,
        # 7. SSIM ✅
        'SSIM':                round(ssim_val, 6) if ssim_val is not None else None,
        # 8. IE ⚠️
        'IE_MAE':              round(ie_val['IE_MAE'],  6) if ie_val['IE_MAE']  is not None else None,
        'IE_RMSE':             round(ie_val['IE_RMSE'], 6) if ie_val['IE_RMSE'] is not None else None,
        # 9. FVD ⚠️ (cần sequence)
        'FVD':                 'N/A (needs video sequence)',
        # 10. Warping Error ⚠️ (cần sequence)
        'Warping_Error':       'N/A (needs video sequence)',
        # 11. Hausdorff Distance ❌
        'Hausdorff_Distance':  'N/A',
        # 12. Line IoU ❌
        'Line_IoU':            'N/A',
        # 13. Disconnection Rate ❌
        'Disconnection_Rate':  'N/A',
        # 14. Arbitrary t Support ✅
        'Arbitrary_t_Support': 'YES — t in (0,inf), both interp & predict',
        # 15. Inference Speed ✅
        'Inference_ms':        round(latency_ms, 3) if latency_ms is not None else None,
        'Inference_FPS':       round(1000.0 / latency_ms, 2) if latency_ms is not None else None,
        # 16. VRAM Peak ✅
        'VRAM_Peak_MB':        round(vram_mb, 2) if vram_mb is not None else None,
        # 17. Fine-tuning Speed ⚠️
        'Finetuning_Speed':    'N/A (not measured in this run)',
    }
    results_list.append(row)

print(f'\n✅ Benchmark hoàn thành! Đã xử lý {len(results_list)} clips')
if error_clips:
    print(f'   ⚠️  {len(error_clips)} clip bị lỗi:')
    for ec in error_clips[:5]:
        print(f'      - {ec["clip"]}: {ec["error"][:100]}')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5D: THỐNG KÊ & TÓM TẮT KẾT QUẢ
# ──────────────────────────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(results_list)

# Các cột số để tính thống kê
numeric_metrics = ['PSNR', 'SSIM', 'LPIPS', 'IE_MAE', 'IE_RMSE',
                   'Inference_ms', 'Inference_FPS', 'VRAM_Peak_MB']

print('=' * 75)
print(f'  📊 BENCHMARK RESULTS SUMMARY — uniVIP ({MODEL_VARIANT})')
print(f'  Dataset: ATD-12K test_2k_540p | Total clips: {len(df)}')
print('=' * 75)

summary_rows = []
for col in numeric_metrics:
    vals = pd.to_numeric(df[col], errors='coerce').dropna()
    if len(vals) > 0:
        direction = '↑' if col in ['PSNR', 'SSIM', 'Inference_FPS'] else '↓' if col in ['LPIPS', 'IE_MAE', 'IE_RMSE', 'Inference_ms'] else '—'
        summary_rows.append({
            'Metric':   f'{col} {direction}',
            'Mean':     f'{vals.mean():.5f}',
            'Std':      f'{vals.std():.5f}',
            'Min':      f'{vals.min():.5f}',
            'Max':      f'{vals.max():.5f}',
            'Count':    int(len(vals)),
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_markdown(index=False))

# Thêm thống kê Arbitrary t
print()
print('=' * 75)
print(f'  Arbitrary t Support: YES (t = 0.25, 0.5, 0.75 đều được hỗ trợ)')
print(f'  Global Inference Speed (warmup-free): {GLOBAL_SPEED_MS:.2f} ms | {GLOBAL_SPEED_FPS:.1f} FPS')
print('=' * 75)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5E: XUẤT FILE CSV KẾT QUẢ
# ──────────────────────────────────────────────────────────────────────────────
csv_filename = f'benchmark_results_uniVIP.csv'
csv_path     = os.path.join(OUTPUT_DIR, csv_filename)

# Đảm bảo thứ tự cột theo đúng quy chuẩn 17 metrics
ORDERED_COLUMNS = [
    # Metadata
    'clip_name', 'model', 'resolution', 'status', 'error_msg',
    # 17 Metrics chuẩn
    'CD',                  # 1
    'WCD',                 # 2
    'LPIPS',               # 3
    'EMD',                 # 4
    'RRLD',                # 5
    'PSNR',                # 6
    'SSIM',                # 7
    'IE_MAE',              # 8a
    'IE_RMSE',             # 8b
    'FVD',                 # 9
    'Warping_Error',       # 10
    'Hausdorff_Distance',  # 11
    'Line_IoU',            # 12
    'Disconnection_Rate',  # 13
    'Arbitrary_t_Support', # 14
    'Inference_ms',        # 15a
    'Inference_FPS',       # 15b
    'VRAM_Peak_MB',        # 16
    'Finetuning_Speed',    # 17
]

df_ordered = df[ORDERED_COLUMNS]
df_ordered.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'✅ Kết quả đã lưu: {csv_path}')
print(f'   Số hàng: {len(df_ordered)}')
print(f'   Số cột:  {len(df_ordered.columns)}')
print()

# Preview đầu file
print('📋 Preview 3 hàng đầu:')
display_cols = ['clip_name', 'PSNR', 'SSIM', 'LPIPS', 'IE_MAE', 'Inference_ms', 'VRAM_Peak_MB']
print(df_ordered[display_cols].head(3).to_markdown(index=False))

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5F: VISUALIZATION — So Sánh Pred vs Ground Truth
# ──────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

if BENCH_VIZ_SAMPLES > 0 and len(viz_frames) > 0:
    n_viz = len(viz_frames)
    fig   = plt.figure(figsize=(18, n_viz * 4.5))
    fig.suptitle(
        f'uniVIP ({MODEL_VARIANT}) — Benchmark Visualization on ATD-12K\n'
        f'Showing {n_viz}/{len(bench_dataset)} clips',
        fontsize=13, fontweight='bold', y=1.01
    )
    
    gs = gridspec.GridSpec(n_viz, 4, figure=fig, hspace=0.5, wspace=0.03)
    
    for i, fd in enumerate(viz_frames):
        images = [
            (fd['img0'],  'Input (frame1, t=0)'),
            (fd['img1'],  'Input (frame3, t=1)'),
            (fd['pred'],  f'Predicted (t=0.5)'),
            (fd['gt'],    'Ground Truth (frame2)'),
        ]
        
        for j, (img, title) in enumerate(images):
            ax = fig.add_subplot(gs[i, j])
            ax.imshow(np.clip(img, 0, 1))
            
            if j == 2:
                title += f'\nPSNR={fd["psnr"]:.2f}dB | SSIM={fd["ssim"]:.4f}'
                title += f'\nLPIPS={fd["lpips"]:.4f}'
            
            ax.set_title(title, fontsize=8, pad=3)
            ax.axis('off')
        
        # Tên clip
        fig.text(
            -0.01, 1 - (i + 0.5) / n_viz,
            f'[{i+1}] {fd["clip"]}',
            va='center', ha='right', fontsize=7, color='#555',
            transform=fig.transFigure, rotation=90
        )
    
    viz_path = os.path.join(OUTPUT_DIR, 'benchmark_uniVIP_visualization.png')
    plt.savefig(viz_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'✅ Visualization đã lưu: {viz_path}')
else:
    print('(Visualization bị tắt hoặc không có frame nào để hiển thị)')

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5G: BÁO CÁO CUỐI — BẢNG 17 CHỈ SỐ CHUẨN HÓA (Markdown Format)
# ──────────────────────────────────────────────────────────────────────────────
import pandas as pd

# Tính mean của các metric số
def safe_mean(series):
    vals = pd.to_numeric(series, errors='coerce').dropna()
    return f'{vals.mean():.5f}' if len(vals) > 0 else 'N/A'

print('\n' + '=' * 75)
print('  📊 FINAL REPORT: 17-METRIC NORMALIZED TABLE')
print(f'  Model: uniVIP-{"L" if USE_LARGE_MODEL else "B"} | Dataset: ATD-12K 540p | Clips: {len(df)}')
print('=' * 75)

report_rows = [
    ('#',  'Metric',               'Status', 'Value (Mean)'),
    ('--', '---',                  '---',    '---'),
    ('1',  'Chamfer Distance (CD)','❌ N/A',  'N/A'),
    ('2',  'Weighted CD (WCD)',    '❌ N/A',  'N/A'),
    ('3',  'LPIPS ↓',              '✅',      safe_mean(df['LPIPS'])),
    ('4',  'EMD',                  '❌ N/A',  'N/A'),
    ('5',  'RRLD',                 '❌ N/A',  'N/A'),
    ('6',  'PSNR ↑ (dB)',         '✅',      safe_mean(df['PSNR'])),
    ('7',  'SSIM ↑',               '✅',      safe_mean(df['SSIM'])),
    ('8',  'IE MAE ↓',            '⚠️',      safe_mean(df['IE_MAE'])),
    ('8b', 'IE RMSE ↓',           '⚠️',      safe_mean(df['IE_RMSE'])),
    ('9',  'FVD',                  '⚠️',      'N/A (needs video seq)'),
    ('10', 'Warping Error',        '⚠️',      'N/A (needs video seq)'),
    ('11', 'Hausdorff Distance',   '❌ N/A',  'N/A'),
    ('12', 'Line IoU',             '❌ N/A',  'N/A'),
    ('13', 'Disconnection Rate',   '❌ N/A',  'N/A'),
    ('14', 'Arbitrary t Support',  '✅',      'YES (t ∈ (0,∞))'),
    ('15a','Inference Speed (ms)', '✅',      f'{GLOBAL_SPEED_MS:.2f} ms'),
    ('15b','Inference Speed (FPS)','✅',      f'{GLOBAL_SPEED_FPS:.1f} FPS'),
    ('16', 'VRAM Peak (MB)',       '✅',      safe_mean(df['VRAM_Peak_MB'])),
    ('17', 'Fine-tuning Speed',    '⚠️',      'N/A (not measured)'),
]

col_widths = [4, 28, 10, 28]
for row in report_rows:
    line = '| '
    for val, width in zip(row, col_widths):
        line += val.ljust(width) + ' | '
    print(line)

print('\n✅ Benchmark hoàn thành!')
print(f'   CSV kết quả: {os.path.join(OUTPUT_DIR, "benchmark_results_uniVIP.csv")}')

---
## CELL 6 — Kiểm Tra Arbitrary t Support (Tính Năng Đặc Biệt của uniVIP)

Chứng minh khả năng sinh frame tại bất kỳ thời điểm nào, bao gồm cả prediction (t > 1).

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 6: KIỂM TRA ARBITRARY t SUPPORT
# Sinh frame tại nhiều mốc t và so sánh trực quan
# ──────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

T_TEST_VALUES = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5]
# t=0.0 và t=1.0 là 2 frames đầu vào
# t=0.5 là interpolation (GT có sẵn)
# t=1.5 là prediction (future frame)

# Dùng sample đầu tiên để test
sample_arb   = dataset[0]
img0_arb     = sample_arb['img0'].unsqueeze(0).to(DEVICE)
img1_arb     = sample_arb['img1'].unsqueeze(0).to(DEVICE)
gt_arb       = sample_arb['gt']  # GT chỉ có ở t=0.5
orig_h_arb   = sample_arb['h']
orig_w_arb   = sample_arb['w']

img0_arb_pad, _, _ = pad_to_divisor(img0_arb, DIVISOR)
img1_arb_pad, _, _ = pad_to_divisor(img1_arb, DIVISOR)

print(f'Clip: {sample_arb["clip"]}')
print(f'Resolution: {orig_w_arb}x{orig_h_arb}')
print(f'Đang sinh frame tại t = {T_TEST_VALUES}...\n')

t_results = {}
for t in T_TEST_VALUES:
    if abs(t - 0.0) < 1e-6:  # t=0 là img0 gốc
        t_results[t] = img0_arb[0].cpu().numpy().transpose(1, 2, 0)[:, :, ::-1]
        print(f'  t={t:.2f}: (Input img0, không inference)')
    elif abs(t - 1.0) < 1e-6:  # t=1 là img1 gốc
        t_results[t] = img1_arb[0].cpu().numpy().transpose(1, 2, 0)[:, :, ::-1]
        print(f'  t={t:.2f}: (Input img1, không inference)')
    else:
        with torch.no_grad():
            try:
                pred_t, _ = ppl.model(img0_arb_pad, img1_arb_pad, time_period=float(t),
                                       skip_pad=True, fixed_pyr_level=BENCH_FIXED_PYR_LVL)
            except TypeError:
                pred_t, _ = ppl.model(img0_arb_pad, img1_arb_pad, time_period=float(t))
        pred_t = pred_t[:, :, :orig_h_arb, :orig_w_arb][0]  # (C,H,W)
        pred_np = pred_t.cpu().numpy().transpose(1, 2, 0)[:, :, ::-1]  # RGB
        t_results[t] = np.clip(pred_np, 0, 1)
        
        # Tính PSNR nếu có GT
        if abs(t - 0.5) < 1e-6:
            psnr_t = compute_psnr(pred_t, gt_arb.to(DEVICE))
            print(f'  t={t:.2f}: PSNR={psnr_t:.2f}dB (GT available)')
        else:
            label = 'interpolation' if 0 < t < 1 else 'PREDICTION (future)'
            print(f'  t={t:.2f}: OK — {label}')

# Visualization
n_t = len(T_TEST_VALUES)
fig, axes = plt.subplots(1, n_t, figsize=(n_t * 3.5, 3.5))
fig.suptitle(f'uniVIP — Arbitrary Time Synthesis\nClip: {sample_arb["clip"]}',
             fontsize=11, fontweight='bold')

for ax, t in zip(axes, T_TEST_VALUES):
    ax.imshow(t_results[t])
    if abs(t - 0.0) < 1e-6:
        title = f't=0.0\n(Input frame0)'
        ax.set_facecolor('#e8f5e9')
    elif abs(t - 1.0) < 1e-6:
        title = f't=1.0\n(Input frame1)'
        ax.set_facecolor('#e8f5e9')
    elif abs(t - 0.5) < 1e-6:
        title = f't=0.5\n(Interpolation ✅)'
    elif t > 1:
        title = f't={t:.2f}\n(Prediction 🔮)'
    else:
        title = f't={t:.2f}\n(Interpolation)'
    ax.set_title(title, fontsize=8)
    ax.axis('off')

plt.tight_layout()
arb_path = os.path.join(OUTPUT_DIR, 'benchmark_uniVIP_arbitrary_t.png')
plt.savefig(arb_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ Arbitrary t visualization đã lưu: {arb_path}')

---
## Tóm Tắt Kết Quả

Sau khi chạy xong toàn bộ notebook, bạn có:

| File | Mô Tả |
|------|-------|
| `benchmark_results_uniVIP.csv` | Bảng kết quả 17 chỉ số đầy đủ (1 hàng = 1 clip) |
| `benchmark_uniVIP_visualization.png` | So sánh Predicted vs GT cho 4 mẫu đầu |
| `benchmark_uniVIP_arbitrary_t.png` | Demo sinh frame tại nhiều mốc t |

### Chỉ Số Đã Đo Được:
- ✅ **PSNR, SSIM, LPIPS**: Core image quality metrics
- ✅ **Arbitrary t Support**: Unique feature of uniVIP
- ✅ **Inference Speed**: ms/frame và FPS (warmup-free)
- ✅ **VRAM Peak**: GPU memory consumption
- ⚠️ **IE (MAE/RMSE)**: Pixel-level interpolation error
- ❌ **CD, WCD, EMD, RRLD, Hausdorff, Line IoU, DR**: N/A (dành cho line art models)
- ⚠️ **FVD, Warping Error**: Cần chạy multi-frame sequence generation

---

### Tài Liệu Tham Khảo:
- [uniVIP Paper (ICASSP 2025)](https://arxiv.org/pdf/2503.02316)
- [UPR-Net (CVPR 2023)](https://github.com/srcn-ivl/UPR-Net)
- [ATD-12K Dataset](https://github.com/lisiyao21/AnimeInterp)
- [LPIPS](https://github.com/richzhang/PerceptualSimilarity)
- [Softmax-Splatting](https://github.com/sniklaus/softmax-splatting)